# IAN 601 — Module 14: Machine Learning with House Prices

Machine learning uses examples to estimate a relationship that can be applied to new cases.

This notebook is a standalone introduction to supervised machine learning through a house-price regression problem. We begin with a familiar algebraic relationship, add realistic noise, then move to a CSV dataset with multiple numeric and categorical features.

**Main path:** algebra → data → features → train/test split → regression → prediction → evaluation → pipeline

## Learning goals

By the end of this notebook, you should be able to:

1. connect a linear equation with a fitted linear model;
2. explain why real data contains noise;
3. load and inspect a modeling dataset;
4. distinguish features from a target;
5. represent one observation as a feature vector;
6. explain why categorical features require numerical encoding;
7. separate data into training and test sets;
8. fit a linear regression model;
9. compare actual and predicted values;
10. interpret MAE, RMSE, and R² at an introductory level;
11. predict a new house price;
12. explain data leakage;
13. explain why preprocessing and modeling can be combined in a pipeline;
14. distinguish regression from classification.

## How to use this notebook

Run the cells from top to bottom the first time so that variables and imported libraries are available when later cells need them.

As you work:

1. read the explanation before the code;
2. predict the result;
3. run the cell;
4. inspect the output carefully;
5. change one value, condition, or option;
6. rerun the cell and explain what changed.

For notebooks that load a CSV from GitHub, the dataset is loaded automatically. If you restart the runtime, run the notebook again from the beginning.

### Expandable syntax guides

Each major modeling section includes a collapsed **Syntax & parameters** box. Open it for scikit-learn signatures, important parameters, model-shape requirements, metric interpretation, related tools, and leakage cautions.

## 1. Start with algebra: y = 2x + 1

When the relationship is known, prediction is simple.

<details>
<summary><strong>Syntax & concepts — arrays and a known linear rule</strong></summary>

<pre><code class="language-python">x = np.arange(0, 11)
y = 2 * x + 1</code></pre>

<p><code>np.arange(start, stop, step)</code> creates evenly spaced values using a step size. The <code>stop</code> value is excluded.</p>

<p>The rule <code>y = 2x + 1</code> has:</p>
<ul>
<li>slope = 2;</li>
<li>intercept = 1.</li>
</ul>

<p>In supervised learning, the model tries to estimate a relationship from observed examples instead of being given the exact rule.</p>
</details>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

x = np.arange(0, 11)
y = 2 * x + 1

pd.DataFrame({"x": x, "y": y})

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(x, y, marker="o")
plt.title("Known Relationship: y = 2x + 1")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

## 2. Reverse engineer the relationship

If we only have the x and y values, a linear model can estimate the slope and intercept.

<details>
<summary><strong>Syntax & parameters — LinearRegression</strong></summary>

<pre><code class="language-python">from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X, y)</code></pre>

<p><strong>Important methods and attributes</strong></p>
<ul>
<li><code>fit(X, y)</code> — estimate model parameters from training data.</li>
<li><code>predict(X_new)</code> — generate predictions.</li>
<li><code>coef_</code> — fitted coefficient(s).</li>
<li><code>intercept_</code> — fitted intercept.</li>
<li><code>score(X, y)</code> — R² score for regression.</li>
</ul>

<p><strong>Common constructor parameter</strong></p>
<pre><code class="language-python">LinearRegression(fit_intercept=True)</code></pre>

<p><code>X</code> must be two-dimensional. A single feature often needs shape <code>(n_rows, 1)</code>.</p>
</details>

In [ ]:
from sklearn.linear_model import LinearRegression

X_simple = x.reshape(-1, 1)

simple_model = LinearRegression()
simple_model.fit(X_simple, y)

print("Estimated slope:", simple_model.coef_[0])
print("Estimated intercept:", simple_model.intercept_)

## 3. Real data is noisy

Real-world measurements rarely fall perfectly on one line.

<details>
<summary><strong>Syntax & parameters — reproducible random noise</strong></summary>

<pre><code class="language-python">rng = np.random.default_rng(601)

noise = rng.normal(
    loc=0,
    scale=5,
    size=30
)</code></pre>

<ul>
<li><code>loc</code> — mean of the normal distribution.</li>
<li><code>scale</code> — standard deviation.</li>
<li><code>size</code> — number or shape of generated values.</li>
</ul>

<p>Using a fixed seed makes the random values reproducible, which is useful for teaching, debugging, and comparing experiments.</p>
</details>

In [ ]:
rng = np.random.default_rng(601)

x_noisy = np.arange(0, 30)
y_noisy = 2 * x_noisy + 1 + rng.normal(0, 5, size=len(x_noisy))

plt.figure(figsize=(7, 4))
plt.scatter(x_noisy, y_noisy)
plt.title("Noisy Observations")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

## 4. Best-fit line

The model estimates the line that best summarizes the relationship in the noisy data.

<details>
<summary><strong>Syntax & concepts — fit(), predict(), and residuals</strong></summary>

<pre><code class="language-python">model.fit(X, y)
predicted = model.predict(X)

residuals = y - predicted</code></pre>

<p>A residual is the difference between an observed value and its prediction.</p>

<p>Linear regression chooses coefficients that minimize the sum of squared residuals.</p>

<p>Useful diagnostic questions:</p>
<ul>
<li>Are residuals generally centered around zero?</li>
<li>Do errors grow as predictions increase?</li>
<li>Are there influential unusual observations?</li>
</ul>
</details>

In [ ]:
model_noisy = LinearRegression()
model_noisy.fit(x_noisy.reshape(-1, 1), y_noisy)

predicted_noisy = model_noisy.predict(x_noisy.reshape(-1, 1))

plt.figure(figsize=(7, 4))
plt.scatter(x_noisy, y_noisy, label="Observed")
plt.plot(x_noisy, predicted_noisy, label="Best-fit line")
plt.title("Observed Data and Best-Fit Line")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.show()

print("Estimated slope:", round(model_noisy.coef_[0], 3))
print("Estimated intercept:", round(model_noisy.intercept_, 3))

## 5. Load the house-price CSV

Now we move from a simple mathematical example to a realistic tabular dataset.

<details>
<summary><strong>Syntax & parameters — loading modeling data</strong></summary>

<pre><code class="language-python">houses = pd.read_csv(data_url)</code></pre>

<p>Useful <code>read_csv()</code> parameters include:</p>
<ul>
<li><code>usecols</code> — load selected columns.</li>
<li><code>dtype</code> — specify data types.</li>
<li><code>na_values</code> — extra missing-value markers.</li>
<li><code>parse_dates</code> — parse date columns.</li>
<li><code>nrows</code> — load only a sample.</li>
</ul>

<p>Before modeling, confirm that each row represents one observation and that the target column is clearly defined.</p>
</details>

In [ ]:
data_url = "https://raw.githubusercontent.com/ali2449/IAN-601-Introduction-to-Data-Analytics-Methods-and-Approaches/main/IAN601_Module14_House_Prices_600.csv"

houses = pd.read_csv(data_url)

houses.head()

## 6. Inspect the housing data

The target is **sale_price**. Everything else that is available before the sale can potentially be used as a feature.

<details>
<summary><strong>Syntax & parameters — modeling data inspection</strong></summary>

<pre><code class="language-python">df.shape
df.head()
df.dtypes
df.info()
df.isna().sum()
df.describe()
df.describe(include="all")</code></pre>

<p>Also inspect category frequencies:</p>
<pre><code class="language-python">df["neighborhood"].value_counts(dropna=False)</code></pre>

<p>Before modeling, look for missing values, impossible values, duplicated IDs, extreme values, and columns that may leak information from the future.</p>
</details>

In [ ]:
print("Shape:", houses.shape)
print()
print(houses.dtypes)
print()
print("Missing values:")
print(houses.isna().sum())

In [ ]:
houses.describe(include="all").T

## 7. Start with one feature: square feet

This is the easiest connection to the algebra idea.

We ask:

> Can square footage alone help predict sale price?

<details>
<summary><strong>Syntax & parameters — scatter plots for modeling</strong></summary>

<pre><code class="language-python">plt.scatter(
    x,
    y,
    alpha=0.5,
    s=30
)</code></pre>

<p>A scatter plot is useful before regression because it can reveal:</p>
<ul>
<li>direction of association;</li>
<li>rough linearity or non-linearity;</li>
<li>clusters;</li>
<li>unusual observations;</li>
<li>changing spread.</li>
</ul>

<p>Linear regression is most natural when the relationship is reasonably approximated by a straight line.</p>
</details>

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(
    houses["square_feet"],
    houses["sale_price"],
    alpha=0.55
)
plt.title("Square Feet vs Sale Price")
plt.xlabel("Square feet")
plt.ylabel("Sale price")
plt.show()

## 8. Fit a one-feature linear regression model

<details>
<summary><strong>Syntax & concepts — one-feature X and target y</strong></summary>

<pre><code class="language-python">X = df[["square_feet"]]
y = df["sale_price"]</code></pre>

<p>Double brackets keep <code>X</code> as a two-dimensional DataFrame. Single brackets return a one-dimensional Series.</p>

<pre><code class="language-python">model = LinearRegression()
model.fit(X, y)</code></pre>

<p>With one feature, the fitted equation has the form:</p>
<pre><code>predicted_price = intercept + coefficient * square_feet</code></pre>
</details>

In [ ]:
X_one = houses[["square_feet"]]
y_price = houses["sale_price"]

one_feature_model = LinearRegression()
one_feature_model.fit(X_one, y_price)

print("Slope:", round(one_feature_model.coef_[0], 2))
print("Intercept:", round(one_feature_model.intercept_, 2))

## 9. Draw the regression line

<details>
<summary><strong>Syntax & parameters — linspace(), reshape(), and prediction grid</strong></summary>

<pre><code class="language-python">x_line = np.linspace(
    minimum,
    maximum,
    100
).reshape(-1, 1)

y_line = model.predict(x_line)</code></pre>

<p><code>np.linspace(start, stop, num)</code> creates a chosen number of evenly spaced values including both endpoints.</p>

<p><code>reshape(-1, 1)</code> converts the one-dimensional values into a two-dimensional feature matrix with one column.</p>
</details>

In [ ]:
x_line = np.linspace(
    houses["square_feet"].min(),
    houses["square_feet"].max(),
    100
).reshape(-1, 1)

y_line = one_feature_model.predict(x_line)

plt.figure(figsize=(7, 4))
plt.scatter(
    houses["square_feet"],
    houses["sale_price"],
    alpha=0.35,
    label="Observed houses"
)
plt.plot(
    x_line,
    y_line,
    linewidth=2,
    label="Regression line"
)

plt.title("One-Feature House Price Model")
plt.xlabel("Square feet")
plt.ylabel("Sale price")
plt.legend()
plt.show()

## 10. Predict the price of a new house using one feature

<details>
<summary><strong>Syntax & parameters — predict() on new data</strong></summary>

<pre><code class="language-python">new_house = pd.DataFrame({
    "square_feet": [1800]
})

prediction = model.predict(new_house)</code></pre>

<p>The new data must have the same feature structure expected by the model.</p>

<p>Using a DataFrame with matching column names reduces mistakes and avoids feature-name warnings.</p>

<p><code>predict()</code> returns an array, even when predicting one row.</p>
</details>

In [ ]:
new_square_feet = pd.DataFrame({
    "square_feet": [1800]
})

predicted_price = one_feature_model.predict(new_square_feet)[0]

print(f"Predicted price for 1,800 sq. ft.: ${predicted_price:,.0f}")

## 11. One feature is useful, but incomplete

House price depends on more than square footage.

We can add:

- bedrooms
- bathrooms
- age
- lot size
- distance to the city center
- neighborhood
- garage
- condition

## 12. Features become a numeric vector

For one house, numeric features can be thought of as a vector.

<details>
<summary><strong>Syntax & concepts — feature matrices and vectors</strong></summary>

<pre><code class="language-python">features = df[[
    "square_feet",
    "bedrooms",
    "bathrooms"
]]

one_vector = features.iloc[0].to_numpy()</code></pre>

<p>Terminology:</p>
<ul>
<li><strong>feature</strong> — one input variable;</li>
<li><strong>feature vector</strong> — feature values for one observation;</li>
<li><strong>feature matrix X</strong> — feature vectors for many observations;</li>
<li><strong>target y</strong> — value the model tries to predict.</li>
</ul>

<p>Many scikit-learn estimators ultimately operate on numerical arrays even when the input begins as a DataFrame.</p>
</details>

In [ ]:
numeric_features = [
    "square_feet",
    "bedrooms",
    "bathrooms",
    "age_years",
    "lot_size_acres",
    "distance_to_center_miles"
]

houses[numeric_features].head()

In [ ]:
first_house_vector = houses.loc[0, numeric_features].to_numpy()

print(first_house_vector)

## 13. What about text features?

Machine learning models need numbers.

Categorical columns such as neighborhood, garage, and condition can be converted with one-hot encoding.

<details>
<summary><strong>Syntax & parameters — one-hot encoding</strong></summary>

<p><strong>Pandas preview</strong></p>
<pre><code class="language-python">pd.get_dummies(
    df[["neighborhood", "garage"]],
    dtype=int,
    drop_first=False
)</code></pre>

<p><strong>scikit-learn encoder</strong></p>
<pre><code class="language-python">OneHotEncoder(
    handle_unknown="ignore"
)</code></pre>

<ul>
<li><code>handle_unknown="ignore"</code> prevents prediction from failing when an unseen category appears.</li>
<li><code>drop="first"</code> can remove one category per feature when desired, though it is not always necessary.</li>
<li><code>sparse_output</code> controls sparse versus dense output in newer scikit-learn versions.</li>
</ul>

<p>Do not assign arbitrary numbers such as North=1, South=2, West=3 unless the numbers have real ordered meaning.</p>
</details>

In [ ]:
categorical_preview = pd.get_dummies(
    houses[["neighborhood", "garage", "condition"]],
    dtype=int
)

categorical_preview.head()

## 14. Define X and y

- **X** contains the input features.
- **y** contains the target we want to predict.

<details>
<summary><strong>Syntax & concepts — separating features and target</strong></summary>

<pre><code class="language-python">X = df[feature_columns]
y = df["sale_price"]</code></pre>

<p>Avoid including:</p>
<ul>
<li>the target itself inside <code>X</code>;</li>
<li>unique identifiers that do not carry useful predictive structure;</li>
<li>information only available after the outcome;</li>
<li>variables that directly encode the answer.</li>
</ul>

<p>Check shapes:</p>
<pre><code class="language-python">print(X.shape)
print(y.shape)</code></pre>
</details>

In [ ]:
feature_columns = [
    "square_feet",
    "bedrooms",
    "bathrooms",
    "age_years",
    "lot_size_acres",
    "distance_to_center_miles",
    "neighborhood",
    "garage",
    "condition"
]

X = houses[feature_columns]
y = houses["sale_price"]

print("X shape:", X.shape)
print("y shape:", y.shape)

## 15. Train/test split

We should not evaluate the model on the same rows used to train it.

The test set represents houses the model did not see during training.

<details>
<summary><strong>Syntax & parameters — train_test_split()</strong></summary>

<pre><code class="language-python">X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=601
)</code></pre>

<ul>
<li><code>test_size</code> — fraction or count reserved for testing.</li>
<li><code>train_size</code> — optional explicit training size.</li>
<li><code>random_state</code> — reproducible random split.</li>
<li><code>shuffle</code> — whether to shuffle before splitting.</li>
<li><code>stratify</code> — preserve class proportions for classification.</li>
</ul>

<p>For time-series prediction, random splitting may be inappropriate because future data should not be used to predict the past.</p>
</details>

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=601
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## 16. Build preprocessing and regression together

A pipeline keeps the preprocessing steps and model together.

Numeric columns are standardized. Categorical columns are one-hot encoded.

<details>
<summary><strong>Syntax & parameters — ColumnTransformer, StandardScaler, OneHotEncoder, Pipeline</strong></summary>

<pre><code class="language-python">preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

model = Pipeline([
    ("preprocess", preprocess),
    ("model", LinearRegression())
])</code></pre>

<p><strong>ColumnTransformer</strong> applies different transformations to different column groups.</p>

<p><strong>StandardScaler</strong> transforms numeric values roughly to mean 0 and standard deviation 1.</p>

<p><strong>OneHotEncoder</strong> converts categorical labels into numeric indicator columns.</p>

<p><strong>Pipeline</strong> ensures the same preprocessing is applied during both fitting and prediction.</p>

<p>Related preprocessing tools include <code>SimpleImputer</code>, <code>MinMaxScaler</code>, and <code>PolynomialFeatures</code>.</p>
</details>

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

numeric_cols = [
    "square_feet",
    "bedrooms",
    "bathrooms",
    "age_years",
    "lot_size_acres",
    "distance_to_center_miles"
]

categorical_cols = [
    "neighborhood",
    "garage",
    "condition"
]

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

house_model = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", LinearRegression())
    ]
)

house_model

## 17. Train the house-price model

<details>
<summary><strong>Syntax & concepts — fitting a pipeline</strong></summary>

<pre><code class="language-python">house_model.fit(
    X_train,
    y_train
)</code></pre>

<p>When a pipeline is fitted:</p>
<ol>
<li>the preprocessing steps learn what they need from <code>X_train</code>;</li>
<li>the training features are transformed;</li>
<li>the final estimator is fitted using the transformed training data.</li>
</ol>

<p>Do not fit preprocessing separately on the full dataset before the train/test split. That can leak information from the test set.</p>
</details>

In [ ]:
house_model.fit(X_train, y_train)

print("Model trained")

## 18. Predict prices for the test set

<details>
<summary><strong>Syntax & concepts — test-set prediction</strong></summary>

<pre><code class="language-python">predictions = model.predict(X_test)</code></pre>

<p>The pipeline automatically applies its fitted preprocessing to <code>X_test</code> before generating predictions.</p>

<p>Create an evaluation table:</p>
<pre><code class="language-python">comparison = pd.DataFrame({
    "actual": y_test.to_numpy(),
    "predicted": predictions,
    "error": y_test.to_numpy() - predictions
})</code></pre>

<p>Inspecting individual errors can reveal patterns that a single overall metric hides.</p>
</details>

In [ ]:
predicted_prices = house_model.predict(X_test)

comparison = pd.DataFrame({
    "actual_price": y_test.to_numpy(),
    "predicted_price": predicted_prices
})

comparison.head(10).round(0)

## 19. Actual vs predicted prices

A good regression model should place many points close to the diagonal line.

<details>
<summary><strong>Syntax & interpretation — actual vs predicted plot</strong></summary>

<pre><code class="language-python">plt.scatter(y_test, predictions)

minimum = min(y_test.min(), predictions.min())
maximum = max(y_test.max(), predictions.max())

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--"
)</code></pre>

<p>The diagonal reference line represents perfect predictions.</p>

<ul>
<li>Point on line → prediction equals actual value.</li>
<li>Point above line → model predicted too high.</li>
<li>Point below line → model predicted too low.</li>
</ul>

<p>Look for systematic patterns rather than only overall closeness.</p>
</details>

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(
    y_test,
    predicted_prices,
    alpha=0.65
)

minimum = min(y_test.min(), predicted_prices.min())
maximum = max(y_test.max(), predicted_prices.max())

plt.plot(
    [minimum, maximum],
    [minimum, maximum],
    linestyle="--"
)

plt.title("Actual vs Predicted House Prices")
plt.xlabel("Actual price")
plt.ylabel("Predicted price")
plt.show()

## 20. Evaluate regression

Three common regression metrics are:

- **MAE** — average absolute prediction error
- **RMSE** — gives more weight to large errors
- **R²** — how much of the variation is explained by the model

<details>
<summary><strong>Syntax & concepts — MAE, RMSE, and R²</strong></summary>

<pre><code class="language-python">from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)</code></pre>

<p><strong>MAE</strong> — average absolute error in the target's original units.</p>

<p><strong>RMSE</strong> — square root of mean squared error; penalizes large errors more heavily.</p>

<p><strong>R²</strong> — fraction of variance explained relative to a mean-only baseline. It can be negative on poor out-of-sample predictions.</p>

<p>No metric is universally best. Choose metrics that match the consequences of prediction errors.</p>
</details>

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, predicted_prices)
rmse = np.sqrt(
    mean_squared_error(y_test, predicted_prices)
)
r2 = r2_score(y_test, predicted_prices)

print(f"MAE:  ${mae:,.0f}")
print(f"RMSE: ${rmse:,.0f}")
print(f"R²:   {r2:.3f}")

## 21. Predict a completely new house

<details>
<summary><strong>Syntax & cautions — prediction for a new observation</strong></summary>

<pre><code class="language-python">new_house = pd.DataFrame({
    "square_feet": [1800],
    "bedrooms": [3],
    ...
})

price = model.predict(new_house)[0]</code></pre>

<p>The new DataFrame should contain the same feature columns used during training.</p>

<p>Predictions are most trustworthy when the new observation is similar to the data used to train the model. Predicting far outside the training range is called extrapolation and can be unreliable.</p>
</details>

In [ ]:
new_house = pd.DataFrame({
    "square_feet": [1800],
    "bedrooms": [3],
    "bathrooms": [2],
    "age_years": [10],
    "lot_size_acres": [0.30],
    "distance_to_center_miles": [7.5],
    "neighborhood": ["West"],
    "garage": ["Yes"],
    "condition": ["Good"]
})

new_prediction = house_model.predict(new_house)[0]

print(f"Predicted sale price: ${new_prediction:,.0f}")

## 22. Compare a simple model with the full model

This helps students see why adding useful features can improve prediction.

<details>
<summary><strong>Syntax & concepts — model comparison</strong></summary>

<p>Compare models on the <strong>same test rows</strong> using the <strong>same metric</strong>.</p>

<pre><code class="language-python">mae_simple = mean_absolute_error(
    y_test,
    simple_predictions
)

mae_full = mean_absolute_error(
    y_test,
    full_predictions
)</code></pre>

<p>A more complex model is not automatically better. Extra features can be useless, redundant, noisy, or unavailable at prediction time.</p>

<p>For serious model selection, cross-validation is preferred over repeatedly tuning choices against one test set.</p>
</details>

In [ ]:
from sklearn.metrics import mean_absolute_error

# One-feature model using only square_feet
simple_train = X_train[["square_feet"]]
simple_test = X_test[["square_feet"]]

simple_house_model = LinearRegression()
simple_house_model.fit(simple_train, y_train)

simple_predictions = simple_house_model.predict(simple_test)

simple_mae = mean_absolute_error(
    y_test,
    simple_predictions
)

full_mae = mean_absolute_error(
    y_test,
    predicted_prices
)

print(f"Square-feet-only MAE: ${simple_mae:,.0f}")
print(f"Full model MAE:       ${full_mae:,.0f}")

## 23. Data leakage

A feature should only contain information that would actually be available when the prediction is made.

For example, do **not** use a value that is calculated after the house is sold to predict sale price.

Leakage can make a model look much better than it really is.

<details>
<summary><strong>Practical checks — avoiding data leakage</strong></summary>

<p>Leakage occurs when training uses information that would not be legitimately available when making a real prediction.</p>

<p><strong>Common leakage examples</strong></p>
<ul>
<li>including the target or a direct derivative of the target as a feature;</li>
<li>using post-sale information to predict sale price;</li>
<li>fitting preprocessing on the full dataset before splitting;</li>
<li>using future observations to predict earlier time periods.</li>
</ul>

<p><strong>Safer pattern</strong></p>
<pre><code class="language-python">X_train, X_test, y_train, y_test = train_test_split(...)

pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)</code></pre>
</details>

## 24. Why use a pipeline?

The pipeline helps ensure that:

1. the same preprocessing is applied during training and prediction,
2. categorical features are handled consistently,
3. preprocessing stays connected to the model,
4. future code is easier to reuse.

<details>
<summary><strong>Syntax & parameters — working with Pipeline</strong></summary>

<pre><code class="language-python">pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", LinearRegression())
])

pipeline.fit(X_train, y_train)
pipeline.predict(X_test)</code></pre>

<p><strong>Inspect named steps</strong></p>
<pre><code class="language-python">pipeline.named_steps["preprocess"]
pipeline.named_steps["model"]</code></pre>

<p><strong>Set nested parameters</strong></p>
<pre><code class="language-python">pipeline.set_params(
    model__fit_intercept=False
)</code></pre>

<p>The double underscore syntax is used to address parameters inside named pipeline steps.</p>
</details>

## 25. Regression versus classification

This module focuses mainly on regression.

- **Regression:** predict a number  
  Example: house price

- **Classification:** predict a category  
  Example: whether a customer will buy

The next machine learning course can explore classification metrics and algorithms in much more depth.

<details>
<summary><strong>Concept guide — regression and classification APIs</strong></summary>

<p>Both tasks often use the same scikit-learn workflow:</p>

<pre><code class="language-python">model.fit(X_train, y_train)
predictions = model.predict(X_test)</code></pre>

<p><strong>Regression</strong> predicts continuous numeric values.</p>
<p>Examples: price, demand, temperature, time.</p>

<p><strong>Classification</strong> predicts class labels.</p>
<p>Examples: yes/no, fraud/not fraud, category A/B/C.</p>

<p>Classification models may also provide class probabilities through methods such as <code>predict_proba()</code>.</p>
</details>

## 26. Machine learning workflow

A useful workflow to remember:

1. Define the question.
2. Load and inspect the data.
3. Choose the target.
4. Choose features.
5. Separate X and y.
6. Split training and testing data.
7. Preprocess features.
8. Train a baseline or simple model.
9. Train a better model.
10. Evaluate on unseen test data.
11. Predict new cases.
12. Check for leakage and limitations.

## Practice challenge

1. Train a model using only `square_feet` and `bedrooms`.
2. Compare its MAE with the full model.
3. Predict the price of a 2,200 sq. ft. house.
4. Change the neighborhood and compare the prediction.
5. Remove `condition` and retrain the model.
6. Explain in your own words what the train/test split protects us from.
7. Give one example of data leakage in a house-price project.

<details>
<summary><strong>Hints & syntax — practice challenge</strong></summary>

<p>Useful tools:</p>
<pre><code class="language-python">train_test_split()
LinearRegression()
Pipeline()
ColumnTransformer()
OneHotEncoder()
StandardScaler()
mean_absolute_error()
mean_squared_error()
r2_score()</code></pre>

<p>Compare models using the same test set. When changing a categorical value such as neighborhood, keep the other house features unchanged so you can isolate the effect on the prediction.</p>
</details>

In [ ]:
# Practice area




## Summary and self-check

The central machine-learning workflow is:

**features → training → prediction → evaluation**

Before finishing, make sure you can explain:

- what `X` and `y` represent;
- why the test set must remain separate from model training;
- why categorical features need encoding;
- what a prediction error means;
- how MAE differs conceptually from RMSE;
- what R² summarizes;
- why a pipeline is useful;
- what data leakage is;
- how regression differs from classification.

The most important idea is not a specific algorithm. It is learning how to build and evaluate a model without using information the model should not have.